# Microsoft GraphRAG (Global Search)

Detect communities in the entity graph, summarise each, and answer global questions with map-reduce.

**Extra requirements:** `pip install networkx`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import networkx as nx

G = nx.Graph()
for i, chunk in enumerate(chunks):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r)
print(G.number_of_nodes(), "entities,", G.number_of_edges(), "edges")

## Community detection
Microsoft uses Leiden; networkx ships Louvain, which behaves similarly for this demo.

In [ ]:
communities = nx.community.louvain_communities(G, seed=42)
communities = [c for c in communities if len(c) >= 3]
print(len(communities), "communities")

## Community reports
Each community's facts are summarised once, at index time.

In [ ]:
reports = []
for c in communities:
    facts = [f"{u} {d['relation']} {v}" for u, v, d in G.subgraph(c).edges(data=True)]
    reports.append(llm("Write a 2-sentence report describing the theme of these facts:\n" + "\n".join(facts)))
for r in reports[:3]: print("-", r)

## Global search = map-reduce
**Map:** each report answers the question with a helpfulness score. **Reduce:** combine the best partial answers.

In [ ]:
def global_search(query):
    partials = []
    for rep in reports:
        out = llm_json(f'Using only this report, answer the question and rate relevance 0-10. Reply as JSON {{"answer": str, "score": int}}.\n\nReport: {rep}\n\nQuestion: {query}')
        if isinstance(out, dict) and out.get("score", 0) >= 5:
            partials.append(out["answer"])
    return llm("Combine these partial answers into one coherent answer:\n- " + "\n- ".join(partials)) if partials else "No relevant communities."

print(global_search("What are the main themes of this handbook?"))